# Notebook 03: Feature Engineering

**Project 4 — End-to-End Machine Learning: LendingClub Loan Default Prediction**

This notebook transforms the cleaned dataset into model-ready features:
1. Separate temporal split index (`issue_d`)
2. Merge FICO into single feature
3. Create derived ratio features
4. Log-transform skewed features
5. Encode categoricals (ordinal, one-hot, target encoding)
6. Drop redundant / near-zero-variance / highly correlated features
7. Scale numerical features
8. Save feature matrix

**Input:** `data/cleaned.parquet`  
**Output:** `data/features.parquet`, `data/temporal_split_index.parquet`

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

print("Libraries loaded.")

Libraries loaded.


## 1. Load & Separate Temporal Split Index

In [2]:
df = pd.read_parquet('data/cleaned.parquet')
print(f"Loaded: {df.shape}")

# Save issue_d for temporal train/test split in Notebook 04
temporal_idx = df[['issue_d']].copy()
temporal_idx.to_parquet('data/temporal_split_index.parquet', index=True)
print(f"Saved temporal_split_index.parquet — range: {df['issue_d'].min()} to {df['issue_d'].max()}")

# Drop issue_d from feature set
df = df.drop(columns=['issue_d'])
print(f"Shape after dropping issue_d: {df.shape}")

Loaded: (1345310, 70)
Saved temporal_split_index.parquet — range: 2007-06-01 00:00:00 to 2018-12-01 00:00:00


Shape after dropping issue_d: (1345310, 69)


## 2. Merge FICO into Single Feature

`fico_range_low` and `fico_range_high` are always exactly 4 apart. Merge into a single `fico_avg` to reduce redundancy.

In [3]:
# Verify they're always 4 apart
diff = df['fico_range_high'] - df['fico_range_low']
print(f"FICO range difference — unique values: {diff.unique()}")

df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2
df = df.drop(columns=['fico_range_low', 'fico_range_high'])
print(f"Created fico_avg — range: [{df['fico_avg'].min()}, {df['fico_avg'].max()}]")
print(f"Shape: {df.shape}")

FICO range difference — unique values: [4. 5.]
Created fico_avg — range: [627.0, 847.5]
Shape: (1345310, 68)


## 3. Create Derived Ratio Features

Domain-driven features capturing financial stress signals:

In [4]:
# Avoid division by zero with small epsilon
eps = 1.0

# Monthly installment as % of monthly income
df['installment_to_income'] = df['installment'] / (df['annual_inc'] / 12 + eps)

# Loan amount relative to annual income
df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'] + eps)

# Revolving balance relative to annual income
df['revol_bal_to_income'] = df['revol_bal'] / (df['annual_inc'] + eps)

# Proportion of accounts currently open
df['open_acc_ratio'] = df['open_acc'] / (df['total_acc'] + eps)

new_features = ['installment_to_income', 'loan_to_income', 'revol_bal_to_income', 'open_acc_ratio']
print("New ratio features:")
print(df[new_features].describe().round(4).to_string())
print(f"\nShape: {df.shape}")

New ratio features:


       installment_to_income  loan_to_income  revol_bal_to_income  open_acc_ratio
count           1.345310e+06    1.345310e+06         1.345310e+06    1.345310e+06
mean            2.209000e-01    4.925700e+00         4.511700e+00    4.756000e-01
std             9.875800e+00    3.329680e+02         6.427868e+02    1.598000e-01
min             1.000000e-04    2.000000e-04         0.000000e+00    0.000000e+00
25%             4.630000e-02    1.247000e-01         9.830000e-02    3.571000e-01
50%             7.220000e-02    2.000000e-01         1.794000e-01    4.615000e-01
75%             1.054000e-01    2.909000e-01         2.955000e-01    5.806000e-01
max             1.466850e+03    4.000000e+04         6.091310e+05    1.555600e+00

Shape: (1345310, 72)


## 4. Log-Transform Skewed Features

`annual_inc`, `revol_bal`, and `tot_cur_bal` have extreme right skew. Adding log-transformed versions improves model performance on linear models.

In [5]:
log_cols = ['annual_inc', 'revol_bal', 'tot_cur_bal']

for col in log_cols:
    new_col = f'log_{col}'
    df[new_col] = np.log1p(df[col])
    print(f"  {new_col}: range [{df[new_col].min():.2f}, {df[new_col].max():.2f}]")

print(f"\nShape: {df.shape}")

  log_annual_inc: range [0.00, 16.21]
  log_revol_bal: range [0.00, 14.88]
  log_tot_cur_bal: range [0.00, 15.89]

Shape: (1345310, 75)


## 5. Encode Categorical Variables

| Feature | Method | Rationale |
|---------|--------|-----------|
| `grade` | Drop (keep sub_grade) | sub_grade is more granular |
| `sub_grade` | Ordinal (A1=1...G5=35) | Natural hierarchy |
| `home_ownership` | Consolidate + one-hot | Low cardinality |
| `purpose` | One-hot | ~13 categories |
| `verification_status` | One-hot | 3 levels |
| `application_type` | One-hot | 2 levels |
| `initial_list_status` | One-hot | 2 levels |
| `addr_state` | Target encoding (smoothed) | 50 levels — one-hot causes overfitting |

In [6]:
# --- sub_grade: ordinal encode A1=1 ... G5=35 ---
subgrade_order = []
for letter in 'ABCDEFG':
    for num in range(1, 6):
        subgrade_order.append(f'{letter}{num}')

subgrade_map = {sg: i + 1 for i, sg in enumerate(subgrade_order)}
df['sub_grade'] = df['sub_grade'].map(subgrade_map)
print(f"sub_grade encoded: range [{df['sub_grade'].min()}, {df['sub_grade'].max()}]")

# Drop grade (sub_grade captures the same info at finer granularity)
df = df.drop(columns=['grade'])
print("Dropped grade (keeping sub_grade)")

sub_grade encoded: range [1, 35]


Dropped grade (keeping sub_grade)


In [7]:
# --- home_ownership: consolidate rare categories ---
print(f"Before consolidation: {df['home_ownership'].value_counts().to_dict()}")

rare_home = ['ANY', 'NONE', 'OTHER']
df['home_ownership'] = df['home_ownership'].replace(rare_home, 'OTHER')
print(f"After consolidation:  {df['home_ownership'].value_counts().to_dict()}")

Before consolidation: {'MORTGAGE': 665579, 'RENT': 534421, 'OWN': 144832, 'ANY': 286, 'OTHER': 144, 'NONE': 48}


After consolidation:  {'MORTGAGE': 665579, 'RENT': 534421, 'OWN': 144832, 'OTHER': 478}


In [8]:
# --- addr_state: smoothed target encoding ---
# Blend per-state default rate with global mean to handle low-count states
global_mean = df['loan_status'].mean()
SMOOTHING = 100  # weight for global mean

state_stats = df.groupby('addr_state')['loan_status'].agg(['mean', 'count'])
state_stats['smoothed'] = (
    (state_stats['count'] * state_stats['mean'] + SMOOTHING * global_mean)
    / (state_stats['count'] + SMOOTHING)
)

print("addr_state target encoding (smoothed):")
print(state_stats.sort_values('smoothed').to_string())

df['addr_state'] = df['addr_state'].map(state_stats['smoothed'])
print(f"\naddr_state encoded — range: [{df['addr_state'].min():.4f}, {df['addr_state'].max():.4f}]")

addr_state target encoding (smoothed):
                mean   count  smoothed
addr_state                            
DC          0.132086    3475  0.133976
ME          0.138424    2030  0.141297
VT          0.139517    2652  0.141702
OR          0.143850   16406  0.144188
NH          0.145759    6449  0.146582
CO          0.155269   29671  0.155418
WV          0.155187    4878  0.156079
WA          0.157565   29188  0.157708
SC          0.162769   15992  0.162998
KS          0.167438   11240  0.167722
WY          0.167693    2922  0.168750
MT          0.168716    3823  0.169504
UT          0.170586   10036  0.170872
CT          0.173763   19728  0.173894
RI          0.178675    5871  0.179026
IL          0.180974   51720  0.181010
WI          0.183510   17732  0.183600
GA          0.183927   43376  0.183963
ID          0.188277    1689  0.188911
MA          0.190528   30977  0.190558
IA          0.142857       7  0.195912
CA          0.196104  196528  0.196106
AZ          0.196299   32

In [9]:
# --- One-hot encode remaining categoricals ---
onehot_cols = ['home_ownership', 'purpose', 'verification_status',
               'application_type', 'initial_list_status']

print("One-hot encoding:")
for col in onehot_cols:
    n_unique = df[col].nunique()
    print(f"  {col}: {n_unique} categories")

df = pd.get_dummies(df, columns=onehot_cols, drop_first=True, dtype=int)
print(f"\nShape after one-hot encoding: {df.shape}")

One-hot encoding:
  home_ownership: 4 categories


  purpose: 14 categories


  verification_status: 3 categories
  application_type: 2 categories
  initial_list_status: 2 categories



Shape after one-hot encoding: (1345310, 89)


## 6. Drop Redundant / Near-Zero-Variance Features

In [10]:
# Identify near-zero variance columns (>99.5% same value)
nzv_cols = []
for col in df.select_dtypes(include=[np.number]).columns:
    if col == 'loan_status':
        continue
    top_pct = df[col].value_counts(normalize=True).iloc[0]
    if top_pct > 0.995:
        nzv_cols.append((col, top_pct))

print(f"Near-zero-variance features (>99.5% single value): {len(nzv_cols)}")
for col, pct in nzv_cols:
    print(f"  {col}: {pct:.4f}")

# Drop them
drop_nzv = [col for col, _ in nzv_cols]
df = df.drop(columns=drop_nzv)
print(f"\nDropped {len(drop_nzv)} features. Shape: {df.shape}")

Near-zero-variance features (>99.5% single value): 8
  acc_now_delinq: 0.9953
  delinq_amnt: 0.9963
  num_tl_120dpd_2m: 0.9993
  num_tl_30dpd: 0.9969
  home_ownership_OTHER: 0.9996
  purpose_educational: 0.9998
  purpose_renewable_energy: 0.9993
  purpose_wedding: 0.9983

Dropped 8 features. Shape: (1345310, 81)


### 6.2 Drop Highly Correlated Features

Remove one feature from each pair with |Pearson r| > 0.90. When choosing which to drop, we keep the feature with higher absolute correlation to the target (`loan_status`). This reduces multicollinearity without losing predictive signal.

Known high-correlation sources:
- Log-transformed features vs their originals (`annual_inc`/`log_annual_inc`, etc.)
- `loan_amnt` vs `installment` (installment is derived from loan amount)
- Overlapping account-count columns

In [11]:
# Compute correlation matrix on numeric features (excluding target)
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != 'loan_status']
corr_matrix = df[numeric_cols].corr().abs()

# Find pairs with |r| > 0.90
THRESHOLD = 0.90
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = []
for col in upper.columns:
    for idx in upper.index:
        val = upper.loc[idx, col]
        if pd.notna(val) and val > THRESHOLD:
            high_corr_pairs.append((idx, col, val))

high_corr_pairs.sort(key=lambda x: -x[2])
print(f"Highly correlated pairs (|r| > {THRESHOLD}): {len(high_corr_pairs)}")
for f1, f2, r in high_corr_pairs:
    print(f"  {f1} <-> {f2}: r = {r:.4f}")

Highly correlated pairs (|r| > 0.9): 6
  open_acc <-> num_sats: r = 0.9838
  num_actv_rev_tl <-> num_rev_tl_bal_gt_0: r = 0.9821
  installment_to_income <-> loan_to_income: r = 0.9800
  int_rate <-> sub_grade: r = 0.9766
  tot_cur_bal <-> tot_hi_cred_lim: r = 0.9730
  loan_amnt <-> installment: r = 0.9534


In [12]:
# For each high-corr pair, drop the one with LOWER absolute correlation to the target
target_corr = df[numeric_cols].corrwith(df['loan_status']).abs()

to_drop = set()
for f1, f2, r in high_corr_pairs:
    if f1 in to_drop or f2 in to_drop:
        continue  # already scheduled for removal
    # Keep the feature more correlated with the target
    if target_corr[f1] >= target_corr[f2]:
        to_drop.add(f2)
        print(f"  Drop '{f2}' (target r={target_corr[f2]:.4f}), keep '{f1}' (target r={target_corr[f1]:.4f})")
    else:
        to_drop.add(f1)
        print(f"  Drop '{f1}' (target r={target_corr[f1]:.4f}), keep '{f2}' (target r={target_corr[f2]:.4f})")

print(f"\nDropping {len(to_drop)} features: {sorted(to_drop)}")
df = df.drop(columns=list(to_drop))
print(f"Shape after high-corr removal: {df.shape}")

  Drop 'num_sats' (target r=0.0269), keep 'open_acc' (target r=0.0281)
  Drop 'num_rev_tl_bal_gt_0' (target r=0.0690), keep 'num_actv_rev_tl' (target r=0.0704)
  Drop 'loan_to_income' (target r=0.0005), keep 'installment_to_income' (target r=0.0011)
  Drop 'int_rate' (target r=0.2588), keep 'sub_grade' (target r=0.2672)
  Drop 'tot_cur_bal' (target r=0.0671), keep 'tot_hi_cred_lim' (target r=0.0745)
  Drop 'installment' (target r=0.0517), keep 'loan_amnt' (target r=0.0656)

Dropping 6 features: ['installment', 'int_rate', 'loan_to_income', 'num_rev_tl_bal_gt_0', 'num_sats', 'tot_cur_bal']
Shape after high-corr removal: (1345310, 75)


## 7. Scale Numerical Features

Apply StandardScaler to all numeric features. In Notebook 04, the scaler will be refit on training data only to avoid leakage.

In [13]:
# Separate target
y = df['loan_status'].copy()
X = df.drop(columns=['loan_status'])

# Identify numeric columns to scale (exclude one-hot binary columns)
binary_cols = [c for c in X.columns if X[c].nunique() <= 2]
numeric_to_scale = [c for c in X.columns if c not in binary_cols]

print(f"Total features: {X.shape[1]}")
print(f"  Numeric to scale: {len(numeric_to_scale)}")
print(f"  Binary (skip scaling): {len(binary_cols)}")

scaler = StandardScaler()
X[numeric_to_scale] = scaler.fit_transform(X[numeric_to_scale])

# Recombine
df_final = pd.concat([X, y], axis=1)
print(f"\nFinal shape: {df_final.shape}")
print(f"Null check: {df_final.isnull().sum().sum()}")

Total features: 74
  Numeric to scale: 57
  Binary (skip scaling): 17



Final shape: (1345310, 75)
Null check: 0


## 8. Save & Summary

In [14]:
df_final.to_parquet('data/features.parquet', index=True)

print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"Final shape: {df_final.shape}")
print(f"Target distribution:")
print(df_final['loan_status'].value_counts())
print(f"\nDefault rate: {df_final['loan_status'].mean():.2%}")
print(f"\nFeature types:")
print(f"  Numeric (scaled): {len(numeric_to_scale)}")
print(f"  Binary (one-hot + flags): {len(binary_cols)}")
print(f"\nAll features ({df_final.shape[1] - 1}):")
print([c for c in df_final.columns if c != 'loan_status'])

FEATURE ENGINEERING SUMMARY
Final shape: (1345310, 75)
Target distribution:
loan_status
0    1076751
1     268559
Name: count, dtype: int64

Default rate: 19.96%

Feature types:
  Numeric (scaled): 57
  Binary (one-hot + flags): 17

All features (74):
['loan_amnt', 'term', 'sub_grade', 'emp_length', 'annual_inc', 'addr_state', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'tot_coll_amt', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_tl_90g_dpd_24m', 'num_tl_o

In [15]:
# Quick verification: reload and check
df_check = pd.read_parquet('data/features.parquet')
print(f"Reload verification — shape: {df_check.shape}, nulls: {df_check.isnull().sum().sum()}")

# Also verify temporal index
t_check = pd.read_parquet('data/temporal_split_index.parquet')
print(f"Temporal index — shape: {t_check.shape}, range: {t_check['issue_d'].min()} to {t_check['issue_d'].max()}")

del df_check, t_check

Reload verification — shape: (1345310, 75), nulls: 0
Temporal index — shape: (1345310, 1), range: 2007-06-01 00:00:00 to 2018-12-01 00:00:00
